# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [3]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [4]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [129]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

Getting the first line of the patents and citations file which is the header

In [130]:
headerP = rddPatents.first()
headerC = rddCitations.first()

In [131]:
headerC

'"CITING","CITED"'

In [132]:
# rddSampledCitations = rddCitations.sample(False, 0.2)
# rddSampledPatents = rddPatents.sample(False, 0.2)

In [133]:
rddSampledCitations.take(5)

['3858242,1515701',
 '3858243,3146465',
 '3858243,3684611',
 '3858244,2635670',
 '3858244,2838924']

In [134]:
rddSampledPatents.take(5)

['3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,',
 '3070806,1963,1096,,"US","PA",,1,,2,6,63,,0,,,,,,,,,',
 '3070813,1963,1096,,"US","NY",,1,,5,6,65,,2,,0,,,,,,,',
 '3070815,1963,1096,,"US","CO",,1,,7,5,59,,1,,0,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

Removing the header and splitting each CSV line into a list of fields for both Patents and Citations file

In [135]:
rddPatentById = rddPatents.filter(lambda x: x != headerP).map(lambda data: (data.split(',')[0], data.split(',')))

In [137]:
rddCitationByCited = rddCitations.filter(lambda x: x != headerC).map(lambda data: (data.split(',')[1], data.split(',')))

### Doing left join of `rddCitationByCited` with `rddPatentById` to get a structure like `rddJoinedPC = CITED -> (CITING, CITED_STATE)`

In [138]:
rddJoinedPC = rddCitationByCited.leftOuterJoin(rddPatentById)

### Caching `rddJoinedPC` to efficiently use in later stages with recomputations

In [139]:
rddJoinedPC.cache()

PythonRDD[232] at RDD at PythonRDD.scala:53

### Filtering the joined result to remove rows which had no matching keys or whose `STATE` field was empty 
### Using map to emit a structure like `rddByCitationAndState = CITING -> (CITED, STATE)`

In [140]:
rddByCitationAndState = rddJoinedPC
                         .filter(lambda x: x[1][1] is not None and x[1][1][5] not in ('', '""'))
                         .map(lambda x: (x[1][0][0],(x[1][0][1], x[1][1][5])))

### Caching `rddByCitationAndState` to efficiently use in later stages with recomputations

In [141]:
rddByCitationAndState.cache()

PythonRDD[233] at RDD at PythonRDD.scala:53

### Doing left join of `rddPatentById` with `rddByCitationAndState` to get a structure like `rddJoined = PATENT -> ([PATENT_DATA..], (CITED, STATE))`

In [142]:
rddJoined = rddPatentById.leftOuterJoin(rddByCitationAndState)

### Filtering out rows which did not have any matching keys or whose `STATE` was empty or the PATENT state did not match with the CITED state

In [144]:
filtered = rddJoined.filter(
    lambda x:
        x[1][1] is not None
        and x[1][0][5] not in ('', '""')
        and x[1][1][1] not in ('', '""')
        and x[1][0][5] == x[1][1][1]
)

### Caching `filtered` to efficiently use in later stages with recomputations

In [145]:
filtered.cache()

PythonRDD[242] at RDD at PythonRDD.scala:53

### Counting the number of self-state citations for each patent. First we assign 1 to every matching citation and then sum the values for each patent ID using reduceByKey

In [146]:
rddCounts = (
    filtered
    .map(lambda x: (x[0], 1))
    .reduceByKey(lambda a, b: a + b)
)

### Left join to get a structure like `rddWithCount = PATENT -> ([PATENT_DATA..], COUNT)`

In [147]:
rddWithCount = rddPatentById.leftOuterJoin(rddCounts)

### Converting the joined data back into the original patent row and appending the self state citation count

In [148]:
finalRDD = rddWithCount.map(
    lambda x: x[1][0] + [
        x[1][1] if x[1][1] is not None else 0
    ]
)

### Sorting all patents by the newly added self state citation count in descending order and take the top 13

In [149]:
top13 = finalRDD.sortBy(
    lambda x: x[-1],
    ascending=False
).take(13)

for row in top13:
    print(row)

['5959466', '1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125]
['5983822', '1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103]
['6008204', '1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100]
['5952345', '1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98]
['5998655', '1999', '14585', '1998', '"US"', '"CA"', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96]
['5958954', '1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96]
['5936426', '1999', '14466', '1997', '"US"', '"CA